# NDVI demostrativo con Sentinel-2 y Google Earth Engine

**Objetivo:** mostrar un flujo sencillo y fácil de explicar:

**Área → Sentinel-2 → filtro de fechas/nubes → mediana → NDVI → mapa → exportación**

> La **mediana** se calcula píxel a píxel usando todas las imágenes seleccionadas. Es útil en una demostración porque genera una imagen representativa del período y reduce la influencia de valores extremos, como nubes residuales.


## 1. Librerías
Carga las librerías necesarias para trabajar con **Google Earth Engine**, visualizar mapas y exportar resultados.


In [ ]:
import ee
import geemap
from google.colab import files

print("✓ Librerías cargadas")


## 2. Parámetros
Define los valores que se pueden modificar fácilmente:

- `PAIS` y `DEPTO`: área de estudio.
- `FECHA_INICIO` y `FECHA_FIN`: período de análisis.
- `MAX_NUBES`: porcentaje máximo de nubosidad permitido.
- `ESCALA_M`: resolución espacial de salida en metros.
- `CRS_SALIDA`: sistema de coordenadas del GeoTIFF.


In [ ]:
PROYECTO_GEE = 'ee-gretapuravida'

PAIS = 'Guatemala'
DEPTO = 'Zacapa'

FECHA_INICIO = '2026-01-01'
FECHA_FIN = '2026-04-01'   # fecha final exclusiva
MAX_NUBES = 20

ESCALA_M = 30
CRS_SALIDA = 'EPSG:32616'


## 3. Conectar con Earth Engine
Inicia la conexión con **Google Earth Engine** utilizando el proyecto definido en `PROYECTO_GEE`.


In [ ]:
try:
    ee.Initialize(project=PROYECTO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)

print("✓ Google Earth Engine conectado")


## 4. Área de estudio
Selecciona el departamento indicado en los parámetros utilizando los límites administrativos de **FAO GAUL**.


In [ ]:
limites = ee.FeatureCollection('FAO/GAUL/2015/level1')

depto = limites.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', PAIS),
        ee.Filter.eq('ADM1_NAME', DEPTO)
    )
)

geom = depto.geometry()
lon, lat = geom.centroid().coordinates().getInfo()

print(f"✓ Área de estudio: {DEPTO}, {PAIS}")


## 5. Sentinel-2 y compuesto por mediana
Selecciona imágenes **Sentinel-2** según el área, las fechas y la nubosidad.

`median()` calcula la mediana de los valores disponibles para cada píxel y genera una sola imagen representativa del período.


In [ ]:
coleccion = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(geom)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_NUBES))
)

print("Número de imágenes:", coleccion.size().getInfo())

# Compuesto representativo del período
s2_mediana = coleccion.median().clip(geom)

print("✓ Compuesto por mediana creado")


## 6. Calcular NDVI
Calcula el **Índice de Vegetación de Diferencia Normalizada (NDVI)** utilizando:

- `B8`: infrarrojo cercano.
- `B4`: rojo.

El NDVI permite representar el vigor o verdor de la vegetación.


In [ ]:
ndvi = s2_mediana.normalizedDifference(['B8', 'B4']).rename('NDVI')

print("✓ NDVI calculado")


## 7. Visualizar
Muestra el compuesto Sentinel-2 y el NDVI sobre **Esri World Imagery**.

Los parámetros `min`, `max` y `palette` controlan cómo se representan los valores en el mapa.


In [ ]:
Map = geemap.Map(
    center=[lat, lon],
    zoom=9,
    basemap='Esri.WorldImagery'
)

Map.addLayer(
    s2_mediana,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Sentinel-2 - mediana'
)

vis_ndvi = {
    'min': 0,
    'max': 0.8,
    'palette': ['red', 'yellow', 'green']
}

Map.addLayer(ndvi, vis_ndvi, 'NDVI')
Map.addLayer(depto.style(color='black', fillColor='00000000'), {}, 'Límite')

Map


## 8. Exportar GeoTIFF
Exporta el NDVI como archivo **GeoTIFF** usando la resolución (`ESCALA_M`) y el sistema de coordenadas (`CRS_SALIDA`) definidos anteriormente.


In [ ]:
ruta = f"/content/NDVI_{DEPTO}.tif"

geemap.ee_export_image(
    ndvi.toFloat(),
    filename=ruta,
    scale=ESCALA_M,
    crs=CRS_SALIDA,
    region=geom,
    file_per_band=False
)

print("✓ GeoTIFF creado:", ruta)


In [ ]:
files.download(ruta)
